# 02 — Preprocessing

This notebook converts the audited hourly air-quality observations into one clean row per selected city and calendar day.

It performs reproducible cleaning, applies the locked study scope, aggregates hourly measurements, completes the city-date calendar, validates the result, and saves `daily_air_quality.csv` for Notebook 03.

No target, lagged feature, data split, or model is created in this notebook.

## 1. Libraries and Locked Preprocessing Configuration

This section imports the required libraries and records the preprocessing decisions approved in Notebook 01.

The notebook will:

- Use the five selected cities.
- Use the common period from 5 August 2022 through 23 November 2025.
- Preserve the recorded source clock times without applying a timezone shift.
- Remove CO2 and non-analytical identifier columns.
- Convert negative NO2 and O3 readings to missing values.
- Aggregate the hourly measurements to one row per city and calendar day.
- Require at least 18 valid hourly AQI readings before assigning daily AQI.
- Create no targets, lagged features, data splits, or models.

In [1]:
from pathlib import Path
from IPython.display import display

import hashlib
import json
import sys

import numpy as np
import pandas as pd

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

STORAGE_ROOT = Path("/content/drive/MyDrive/CSE437_air_quality_group_18")

PATHS = {
    "raw": STORAGE_ROOT / "raw",
    "processed": STORAGE_ROOT / "processed",
}

PATHS["processed"].mkdir(parents=True, exist_ok=True)

SELECTED_CITIES = [
    "Dhaka",
    "Dinājpur",
    "Bherāmāra",
    "Bhola",
    "Cox’s Bāzār",
]

COMMON_START = pd.Timestamp("2022-08-05")
COMMON_END = pd.Timestamp("2025-11-23")

EXPECTED_DATES_PER_CITY = 1207
EXPECTED_DAILY_ROWS = len(SELECTED_CITIES) * EXPECTED_DATES_PER_CITY
EXPECTED_HOURLY_ROWS = EXPECTED_DAILY_ROWS * 24
MIN_VALID_AQI_HOURS = 18

POLLUTANTS = ["pm10", "pm25", "co", "no2", "so2", "o3"]

RAW_FILE = PATHS["raw"] / "AQI Bangladesh.csv"
AUDIT_SUMMARY_FILE = (
    PATHS["processed"]
    / "notebook_01_audit"
    / "audit_summary.json"
)

OUTPUT_FILE = PATHS["processed"] / "daily_air_quality.csv"
SUMMARY_FILE = (
    PATHS["processed"]
    / "notebook_02_preprocessing_summary.json"
)

print("Storage root:", STORAGE_ROOT)
print("Raw input:", RAW_FILE)
print("Audit summary:", AUDIT_SUMMARY_FILE)
print("Daily output:", OUTPUT_FILE)

Mounted at /content/drive
Storage root: /content/drive/MyDrive/CSE437_air_quality_group_18
Raw input: /content/drive/MyDrive/CSE437_air_quality_group_18/raw/AQI Bangladesh.csv
Audit summary: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/notebook_01_audit/audit_summary.json
Daily output: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/daily_air_quality.csv


## 2. Load and Verify the Notebook 01 Decisions

Notebook 02 depends on the completed audit. This section loads `audit_summary.json` and checks that its selected cities, common dates, AQI rule, and CO2 decision match the locked project configuration.

Execution stops if the audit output is missing or inconsistent. This prevents preprocessing from silently using a different study scope.

In [2]:
if not RAW_FILE.is_file():
    raise FileNotFoundError(f"Missing raw AQI file: {RAW_FILE}")

if not AUDIT_SUMMARY_FILE.is_file():
    raise FileNotFoundError(
        "Notebook 01 audit summary was not found. "
        "Run Notebook 01 successfully before continuing."
    )

with AUDIT_SUMMARY_FILE.open("r", encoding="utf-8") as file:
    audit_summary = json.load(file)

assert audit_summary["selected_cities"] == SELECTED_CITIES
assert pd.Timestamp(audit_summary["common_first_date"]) == COMMON_START
assert pd.Timestamp(audit_summary["common_last_date"]) == COMMON_END
assert int(audit_summary["common_usable_days"]) == EXPECTED_DATES_PER_CITY
assert audit_summary["daily_aqi_aggregation"] == "Maximum provided hourly AQI"
assert audit_summary["excluded_predictor"] == "CO2"

decision_table = pd.DataFrame({
    "item": [
        "Selected cities",
        "Common start",
        "Common end",
        "Dates per city",
        "Daily AQI",
        "Minimum valid AQI hours",
        "Excluded variable",
    ],
    "decision": [
        ", ".join(SELECTED_CITIES),
        COMMON_START.date(),
        COMMON_END.date(),
        EXPECTED_DATES_PER_CITY,
        "Maximum supplied hourly AQI",
        MIN_VALID_AQI_HOURS,
        "CO2",
    ],
})

display(decision_table)
print("Notebook 01 decisions verified.")

,item,decision
0,Selected cities,"Dhaka, Dinājpur, Bherāmāra, Bhola, Cox’s Bāzār"
1,Common start,2022-08-05
2,Common end,2025-11-23
3,Dates per city,1207
4,Daily AQI,Maximum supplied hourly AQI
5,Minimum valid AQI hours,18
6,Excluded variable,CO2


Notebook 01 decisions verified.


## 3. Load and Verify the Raw AQI File

This section reads the untouched `AQI Bangladesh.csv` file.

The SHA-256 checksum and dataset dimensions are compared with the values recorded by Notebook 01. This confirms that Notebook 02 is processing the same source file that passed the audit.

In [3]:
def calculate_sha256(file_path):
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


source_checksum = calculate_sha256(RAW_FILE)
df_raw = pd.read_csv(RAW_FILE, low_memory=False)

source_rows, source_columns = df_raw.shape

assert source_checksum == audit_summary["sha256"]
assert source_rows == int(audit_summary["source_rows"])
assert source_columns == int(audit_summary["source_columns"])

print("Source file:", RAW_FILE.name)
print(f"Rows: {source_rows:,}")
print(f"Columns: {source_columns}")
print("SHA-256:", source_checksum)

display(df_raw.head())

Source file: AQI Bangladesh.csv
Rows: 1,048,551
Columns: 13
SHA-256: 8760175fc048eea4180b828fd60d10cb799a73a5144a7e4aca19ddbaf8dbdd62


,city_id,city_name,lat,lon,datetime,pm10,pm2_5,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,aqi
0,7701354,Azimpur,23.7298,90.3854,2022-08-05T00:00,25.0,16.7,252.0,NaN,18.6,5.2,13.0,50.0
1,7701354,Azimpur,23.7298,90.3854,2022-08-05T01:00,18.5,12.3,249.0,NaN,18.5,5.7,19.0,50.0
2,7701354,Azimpur,23.7298,90.3854,2022-08-05T02:00,16.1,10.9,244.0,NaN,18.4,6.5,27.0,51.0
3,7701354,Azimpur,23.7298,90.3854,2022-08-05T03:00,15.4,10.3,233.0,NaN,17.2,6.9,39.0,51.0
4,7701354,Azimpur,23.7298,90.3854,2022-08-05T04:00,16.8,11.3,217.0,NaN,14.8,6.2,51.0,51.0


## 4. Standardize the Audited Schema and Parse Timestamps

The source columns are renamed to short, consistent analytical names using the mapping verified in Notebook 01.

The recorded timestamp text is parsed without using `utc=True` and without adding or subtracting any timezone offset. Therefore, each observation remains assigned to its recorded calendar day.

Numeric conversion failures and timestamp parsing failures are reported before preprocessing continues.

In [4]:
SOURCE_TO_STANDARD = {
    "city_id": "city_id",
    "city_name": "city",
    "lat": "latitude",
    "lon": "longitude",
    "datetime": "timestamp",
    "pm10": "pm10",
    "pm2_5": "pm25",
    "carbon_monoxide": "co",
    "carbon_dioxide": "co2",
    "nitrogen_dioxide": "no2",
    "sulphur_dioxide": "so2",
    "ozone": "o3",
    "aqi": "aqi",
}

missing_source_columns = [
    column for column in SOURCE_TO_STANDARD
    if column not in df_raw.columns
]

if missing_source_columns:
    raise ValueError(f"Missing source columns: {missing_source_columns}")

hourly_all = (
    df_raw[list(SOURCE_TO_STANDARD)]
    .rename(columns=SOURCE_TO_STANDARD)
    .copy()
)

del df_raw

hourly_all["city"] = hourly_all["city"].astype("string").str.strip()
hourly_all.loc[hourly_all["city"].eq(""), "city"] = pd.NA

timestamp_text = hourly_all["timestamp"].astype("string")

try:
    parsed_timestamp = pd.to_datetime(
        timestamp_text,
        errors="coerce",
        format="mixed",
    )
except ValueError:
    parsed_timestamp = pd.to_datetime(timestamp_text, errors="coerce")

timestamp_parse_failures = int(
    (timestamp_text.notna() & parsed_timestamp.isna()).sum()
)

hourly_all["timestamp"] = parsed_timestamp

numeric_columns = [
    "city_id",
    "latitude",
    "longitude",
    "pm10",
    "pm25",
    "co",
    "co2",
    "no2",
    "so2",
    "o3",
    "aqi",
]

numeric_parse_failures = {}

for column in numeric_columns:
    original_non_missing = hourly_all[column].notna()
    converted = pd.to_numeric(hourly_all[column], errors="coerce")

    numeric_parse_failures[column] = int(
        (original_non_missing & converted.isna()).sum()
    )

    hourly_all[column] = converted

parse_summary = pd.DataFrame({
    "column": ["timestamp", *numeric_columns],
    "parse_failures": [
        timestamp_parse_failures,
        *[numeric_parse_failures[column] for column in numeric_columns],
    ],
})

display(parse_summary)

assert len(hourly_all) == source_rows
assert hourly_all["city"].notna().all()
assert timestamp_parse_failures == 0
assert sum(numeric_parse_failures.values()) == 0
assert hourly_all["timestamp"].dt.tz is None

print("Schema and recorded timestamps standardized successfully.")

,column,parse_failures
0,timestamp,0
1,city_id,0
2,latitude,0
3,longitude,0
4,pm10,0
5,pm25,0
6,co,0
7,co2,0
8,no2,0
9,so2,0


Schema and recorded timestamps standardized successfully.


## 5. Convert Invalid Pollutant Readings to Missing

Notebook 01 found one negative NO2 reading and eleven negative O3 readings in the complete source file.

Negative concentrations are invalid, so they are replaced with missing values. Entire rows are not removed because their other measurements may still be valid. The affected rows are displayed before correction so the transformation remains traceable.

In [5]:
negative_no2_mask = hourly_all["no2"].lt(0)
negative_o3_mask = hourly_all["o3"].lt(0)
invalid_pollutant_mask = negative_no2_mask | negative_o3_mask

invalid_pollutant_rows = hourly_all.loc[
    invalid_pollutant_mask,
    ["city", "timestamp", "no2", "o3"],
].copy()

invalid_counts = {
    "negative_no2_to_missing": int(negative_no2_mask.sum()),
    "negative_o3_to_missing": int(negative_o3_mask.sum()),
    "affected_rows": int(invalid_pollutant_mask.sum()),
}

display(invalid_pollutant_rows)
display(pd.Series(invalid_counts, name="count").to_frame())

assert invalid_counts["negative_no2_to_missing"] == int(
    audit_summary["negative_no2_rows"]
)
assert invalid_counts["negative_o3_to_missing"] == int(
    audit_summary["negative_o3_rows"]
)

hourly_all.loc[negative_no2_mask, "no2"] = np.nan
hourly_all.loc[negative_o3_mask, "o3"] = np.nan

assert not hourly_all["no2"].dropna().lt(0).any()
assert not hourly_all["o3"].dropna().lt(0).any()

print("Invalid NO2 and O3 readings converted to missing.")

,city,timestamp,no2,o3
58576,Baniachang,2022-08-29 16:00:00,84.8,-3.0
58577,Baniachang,2022-08-29 17:00:00,83.7,-2.0
348496,Bājitpur,2022-08-29 16:00:00,73.6,-2.0
348497,Bājitpur,2022-08-29 17:00:00,72.5,-2.0
348500,Bājitpur,2022-08-29 20:00:00,67.8,-1.0
348523,Bājitpur,2022-08-30 19:00:00,71.8,-1.0
348524,Bājitpur,2022-08-30 20:00:00,69.1,-1.0
377360,Bāndarban,2022-08-24 08:00:00,-0.1,63.0
981427,Gafargaon,2022-08-30 19:00:00,58.5,-1.0
981428,Gafargaon,2022-08-30 20:00:00,60.0,-1.0


,count
negative_no2_to_missing,1
negative_o3_to_missing,11
affected_rows,12


Invalid NO2 and O3 readings converted to missing.


## 6. Apply the Locked City and Date Scope

Only the five approved cities and the common period are retained.

The end-date filter includes all 24 recorded hours of 23 November 2025. The data are then sorted by city and timestamp.

The following columns are excluded:

- `co2`, because it is approximately 74% missing and is not an approved predictor.
- `city_id`, latitude, and longitude, because they are static identifiers rather than hourly air-quality measurements.

City and date information are retained for grouping and later city-level evaluation.

In [6]:
period_end_exclusive = COMMON_END + pd.Timedelta(days=1)

scope_mask = (
    hourly_all["city"].isin(SELECTED_CITIES)
    & hourly_all["timestamp"].ge(COMMON_START)
    & hourly_all["timestamp"].lt(period_end_exclusive)
)

hourly = hourly_all.loc[scope_mask].copy()
rows_excluded_by_scope = len(hourly_all) - len(hourly)

del hourly_all

hourly["date"] = hourly["timestamp"].dt.normalize()
hourly["hour"] = hourly["timestamp"].dt.floor("h")

hourly = hourly.drop(
    columns=["city_id", "latitude", "longitude", "co2"]
)

hourly = (
    hourly
    .sort_values(["city", "timestamp"])
    .reset_index(drop=True)
)

duplicate_city_timestamps = int(
    hourly.duplicated(["city", "timestamp"]).sum()
)

hourly_rows_by_city = (
    hourly.groupby("city")
    .agg(
        hourly_rows=("timestamp", "size"),
        unique_timestamps=("timestamp", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        calendar_dates=("date", "nunique"),
    )
    .reindex(SELECTED_CITIES)
    .reset_index()
)

display(hourly_rows_by_city)

assert set(hourly["city"].unique()) == set(SELECTED_CITIES)
assert len(hourly) == EXPECTED_HOURLY_ROWS
assert duplicate_city_timestamps == 0
assert hourly["timestamp"].min() == COMMON_START
assert hourly["timestamp"].max() == COMMON_END + pd.Timedelta(hours=23)
assert not hourly[["no2", "o3"]].lt(0).any().any()
assert "co2" not in hourly.columns

print(f"Selected hourly rows: {len(hourly):,}")
print(f"Rows outside the locked scope: {rows_excluded_by_scope:,}")
print("Duplicate city-timestamp pairs:", duplicate_city_timestamps)

,city,hourly_rows,unique_timestamps,first_timestamp,last_timestamp,calendar_dates
0,Dhaka,28968,28968,2022-08-05,2025-11-23 23:00:00,1207
1,Dinājpur,28968,28968,2022-08-05,2025-11-23 23:00:00,1207
2,Bherāmāra,28968,28968,2022-08-05,2025-11-23 23:00:00,1207
3,Bhola,28968,28968,2022-08-05,2025-11-23 23:00:00,1207
4,Cox’s Bāzār,28968,28968,2022-08-05,2025-11-23 23:00:00,1207


Selected hourly rows: 144,840
Rows outside the locked scope: 903,711
Duplicate city-timestamp pairs: 0


## 7. Inspect the Selected Hourly Measurements

Before aggregation, this section reports missing values, valid observations, and observed value ranges for each pollutant and AQI.

It also checks the number of recorded hours available for every city-date. These results provide evidence for the daily aggregation and missing-value decisions.

In [7]:
measurement_columns = POLLUTANTS + ["aqi"]

hourly_quality = pd.DataFrame({
    "missing_values": hourly[measurement_columns].isna().sum(),
    "valid_values": hourly[measurement_columns].notna().sum(),
    "missing_percent": (
        hourly[measurement_columns].isna().mean().mul(100)
    ),
    "minimum": hourly[measurement_columns].min(),
    "maximum": hourly[measurement_columns].max(),
}).round(3)

hours_per_city_date = (
    hourly.groupby(["city", "date"])
    .agg(
        observed_rows=("timestamp", "size"),
        observed_hours=("hour", "nunique"),
    )
    .reset_index()
)

hourly_coverage = (
    hours_per_city_date.groupby("city")
    .agg(
        observed_dates=("date", "size"),
        minimum_hours=("observed_hours", "min"),
        median_hours=("observed_hours", "median"),
        maximum_hours=("observed_hours", "max"),
    )
    .reindex(SELECTED_CITIES)
    .reset_index()
)

display(hourly_quality)
display(hourly_coverage)

,missing_values,valid_values,missing_percent,minimum,maximum
pm10,0,144840,0.0,0.3,451.7
pm25,0,144840,0.0,0.2,314.8
co,0,144840,0.0,58.0,4313.0
no2,0,144840,0.0,0.0,178.0
so2,0,144840,0.0,0.0,99.8
o3,0,144840,0.0,0.0,326.0
aqi,0,144840,0.0,12.0,275.0


,city,observed_dates,minimum_hours,median_hours,maximum_hours
0,Dhaka,1207,24,24.0,24
1,Dinājpur,1207,24,24.0,24
2,Bherāmāra,1207,24,24.0,24
3,Bhola,1207,24,24.0,24
4,Cox’s Bāzār,1207,24,24.0,24


## 8. Aggregate Hourly Measurements to Daily Values

The following daily aggregation rules are used:

- Each pollutant is summarized using its daily **mean** and **maximum**.
- The mean represents the day's overall pollution level.
- The maximum preserves information about short high-pollution periods.
- Missing pollutant readings are not imputed. Statistics are calculated from the available valid readings.
- A valid-hour count is retained for every pollutant so later notebooks can evaluate measurement coverage.
- Daily AQI is the maximum supplied hourly AQI, not a newly calculated regulatory AQI.
- Daily AQI is retained only when at least 18 hourly AQI readings are valid.

No target, lag, rolling feature, split, or model is created here.

In [8]:
daily_aggregations = {
    "observed_rows": ("timestamp", "size"),
    "observed_hours": ("hour", "nunique"),
}

for pollutant in POLLUTANTS:
    daily_aggregations[f"{pollutant}_mean"] = (pollutant, "mean")
    daily_aggregations[f"{pollutant}_max"] = (pollutant, "max")
    daily_aggregations[f"{pollutant}_valid_hours"] = (pollutant, "count")

daily_aggregations["daily_aqi_unchecked"] = ("aqi", "max")
daily_aggregations["aqi_valid_hours"] = ("aqi", "count")

daily = (
    hourly.groupby(["city", "date"])
    .agg(**daily_aggregations)
    .reset_index()
)

daily["daily_aqi"] = daily["daily_aqi_unchecked"].where(
    daily["aqi_valid_hours"].ge(MIN_VALID_AQI_HOURS)
)

daily = daily.drop(columns="daily_aqi_unchecked")

ordered_columns = [
    "city",
    "date",
    "observed_rows",
    "observed_hours",
    "daily_aqi",
    "aqi_valid_hours",
]

for pollutant in POLLUTANTS:
    ordered_columns.extend([
        f"{pollutant}_mean",
        f"{pollutant}_max",
        f"{pollutant}_valid_hours",
    ])

daily = daily[ordered_columns]
observed_daily_rows = len(daily)

print(f"Observed city-date rows before reindexing: {observed_daily_rows:,}")
display(daily.head(10))

Observed city-date rows before reindexing: 6,035


,city,date,observed_rows,observed_hours,daily_aqi,aqi_valid_hours,pm10_mean,pm10_max,pm10_valid_hours,pm25_mean,...,co_valid_hours,no2_mean,no2_max,no2_valid_hours,so2_mean,so2_max,so2_valid_hours,o3_mean,o3_max,o3_valid_hours
0,Bherāmāra,2022-08-05,24,24,53.0,24,18.466667,28.2,24,12.500000,...,24,16.604167,29.4,24,1.275000,3.2,24,36.958333,84.0,24
1,Bherāmāra,2022-08-06,24,24,52.0,24,18.358333,26.4,24,12.629167,...,24,17.575000,33.5,24,1.433333,4.4,24,30.916667,72.0,24
2,Bherāmāra,2022-08-07,24,24,51.0,24,16.862500,23.6,24,11.491667,...,24,16.641667,41.5,24,0.937500,1.4,24,49.958333,106.0,24
3,Bherāmāra,2022-08-08,24,24,48.0,24,11.512500,16.9,24,7.704167,...,24,8.487500,16.9,24,0.958333,1.5,24,53.916667,86.0,24
4,Bherāmāra,2022-08-09,24,24,34.0,24,8.900000,14.3,24,5.975000,...,24,6.179167,12.1,24,0.887500,2.2,24,58.166667,80.0,24
5,Bherāmāra,2022-08-10,24,24,34.0,24,8.254167,14.2,24,5.662500,...,24,6.554167,11.8,24,1.120833,2.0,24,58.625000,82.0,24
6,Bherāmāra,2022-08-11,24,24,53.0,24,20.529167,38.8,24,14.266667,...,24,24.258333,50.4,24,3.770833,12.9,24,31.958333,56.0,24
7,Bherāmāra,2022-08-12,24,24,67.0,24,23.691667,39.8,24,16.391667,...,24,20.079167,47.9,24,4.479167,16.2,24,50.041667,109.0,24
8,Bherāmāra,2022-08-13,24,24,59.0,24,14.662500,22.1,24,10.066667,...,24,10.925000,17.7,24,2.120833,4.0,24,56.750000,96.0,24
9,Bherāmāra,2022-08-14,24,24,42.0,24,10.366667,12.7,24,7.012500,...,24,8.066667,14.8,24,1.066667,2.3,24,48.958333,67.0,24


## 9. Create the Complete City-Date Calendar

Every selected city is reindexed to every calendar date from 5 August 2022 through 23 November 2025.

This step is required before Notebook 03 creates the next-day target. If a date is absent from the hourly source, it must remain visible rather than allowing the next observed row to be mistaken for the next calendar day.

Inserted calendar dates receive zero observation counts while their measurement values remain missing.

In [9]:
full_calendar = pd.date_range(
    COMMON_START,
    COMMON_END,
    freq="D",
)

full_city_date_index = pd.MultiIndex.from_product(
    [SELECTED_CITIES, full_calendar],
    names=["city", "date"],
)

daily_complete = (
    daily.set_index(["city", "date"])
    .reindex(full_city_date_index)
    .reset_index()
)

count_columns = [
    "observed_rows",
    "observed_hours",
    "aqi_valid_hours",
    *[f"{pollutant}_valid_hours" for pollutant in POLLUTANTS],
]

daily_complete[count_columns] = (
    daily_complete[count_columns]
    .fillna(0)
    .astype("int64")
)

daily_complete = (
    daily_complete
    .sort_values(["city", "date"])
    .reset_index(drop=True)
)

inserted_calendar_days = len(daily_complete) - observed_daily_rows

print(f"Complete calendar rows: {len(daily_complete):,}")
print("Calendar rows inserted:", inserted_calendar_days)

display(daily_complete.head())

Complete calendar rows: 6,035
Calendar rows inserted: 0


,city,date,observed_rows,observed_hours,daily_aqi,aqi_valid_hours,pm10_mean,pm10_max,pm10_valid_hours,pm25_mean,...,co_valid_hours,no2_mean,no2_max,no2_valid_hours,so2_mean,so2_max,so2_valid_hours,o3_mean,o3_max,o3_valid_hours
0,Bherāmāra,2022-08-05,24,24,53.0,24,18.466667,28.2,24,12.500000,...,24,16.604167,29.4,24,1.275000,3.2,24,36.958333,84.0,24
1,Bherāmāra,2022-08-06,24,24,52.0,24,18.358333,26.4,24,12.629167,...,24,17.575000,33.5,24,1.433333,4.4,24,30.916667,72.0,24
2,Bherāmāra,2022-08-07,24,24,51.0,24,16.862500,23.6,24,11.491667,...,24,16.641667,41.5,24,0.937500,1.4,24,49.958333,106.0,24
3,Bherāmāra,2022-08-08,24,24,48.0,24,11.512500,16.9,24,7.704167,...,24,8.487500,16.9,24,0.958333,1.5,24,53.916667,86.0,24
4,Bherāmāra,2022-08-09,24,24,34.0,24,8.900000,14.3,24,5.975000,...,24,6.179167,12.1,24,0.887500,2.2,24,58.166667,80.0,24


## 10. Validate the Processed Daily Dataset

This section compares the hourly input with the completed daily output and performs the preprocessing acceptance checks.

The assertions verify:

- Five approved cities are present.
- Every city contains exactly 1,207 calendar dates.
- The output contains exactly one row per city-date.
- Dates are complete and sorted within every city.
- AQI is missing whenever fewer than 18 valid AQI hours are available.
- Observation and valid-hour counts are internally consistent.
- CO2 and static identifiers are absent.
- No target, lagged feature, rolling feature, split, or model output was created.

In [10]:
coverage_by_city = (
    daily_complete.groupby("city")
    .agg(
        calendar_days=("date", "size"),
        recorded_days=(
            "observed_hours",
            lambda values: int(values.gt(0).sum()),
        ),
        usable_aqi_days=(
            "daily_aqi",
            lambda values: int(values.notna().sum()),
        ),
        minimum_observed_hours=("observed_hours", "min"),
        median_observed_hours=("observed_hours", "median"),
        maximum_observed_hours=("observed_hours", "max"),
    )
    .reindex(SELECTED_CITIES)
    .reset_index()
)

daily_missingness = pd.DataFrame({
    "missing_count": daily_complete.isna().sum(),
    "missing_percent": daily_complete.isna().mean().mul(100),
}).round(3)

transformation_summary = pd.DataFrame({
    "stage": [
        "Raw source",
        "Selected hourly scope",
        "Observed daily aggregation",
        "Completed city-date calendar",
    ],
    "rows": [
        source_rows,
        len(hourly),
        observed_daily_rows,
        len(daily_complete),
    ],
})

display(transformation_summary)
display(coverage_by_city)
display(daily_missingness)

assert len(daily_complete) == EXPECTED_DAILY_ROWS
assert set(daily_complete["city"]) == set(SELECTED_CITIES)
assert daily_complete["date"].min() == COMMON_START
assert daily_complete["date"].max() == COMMON_END
assert not daily_complete.duplicated(["city", "date"]).any()

sorted_city_dates = (
    daily_complete[["city", "date"]]
    .sort_values(["city", "date"])
    .reset_index(drop=True)
)

assert daily_complete[["city", "date"]].equals(sorted_city_dates)

for city in SELECTED_CITIES:
    city_dates = daily_complete.loc[
        daily_complete["city"].eq(city),
        "date",
    ].tolist()

    assert city_dates == list(full_calendar)

assert coverage_by_city["calendar_days"].eq(
    EXPECTED_DATES_PER_CITY
).all()

for valid_column in count_columns[2:]:
    assert daily_complete[valid_column].le(
        daily_complete["observed_rows"]
    ).all()

low_aqi_coverage = daily_complete["aqi_valid_hours"].lt(
    MIN_VALID_AQI_HOURS
)

assert daily_complete.loc[
    low_aqi_coverage,
    "daily_aqi",
].isna().all()

assert daily_complete.loc[
    ~low_aqi_coverage,
    "daily_aqi",
].notna().all()

for pollutant in POLLUTANTS:
    valid_hours = daily_complete[f"{pollutant}_valid_hours"]
    mean_values = daily_complete[f"{pollutant}_mean"]
    max_values = daily_complete[f"{pollutant}_max"]

    assert mean_values.loc[valid_hours.eq(0)].isna().all()
    assert max_values.loc[valid_hours.eq(0)].isna().all()
    assert mean_values.loc[valid_hours.gt(0)].notna().all()
    assert max_values.loc[valid_hours.gt(0)].notna().all()

excluded_columns = {
    "city_id",
    "latitude",
    "longitude",
    "timestamp",
    "hour",
    "co2",
}

assert excluded_columns.isdisjoint(daily_complete.columns)

forbidden_terms = [
    "next_day",
    "target",
    "label",
    "lag_",
    "rolling",
    "train",
    "validation",
    "test",
    "prediction",
]

assert not any(
    term in column.lower()
    for column in daily_complete.columns
    for term in forbidden_terms
)

print("All Notebook 02 validation checks passed.")

,stage,rows
0,Raw source,1048551
1,Selected hourly scope,144840
2,Observed daily aggregation,6035
3,Completed city-date calendar,6035


,city,calendar_days,recorded_days,usable_aqi_days,minimum_observed_hours,median_observed_hours,maximum_observed_hours
0,Dhaka,1207,1207,1207,24,24.0,24
1,Dinājpur,1207,1207,1207,24,24.0,24
2,Bherāmāra,1207,1207,1207,24,24.0,24
3,Bhola,1207,1207,1207,24,24.0,24
4,Cox’s Bāzār,1207,1207,1207,24,24.0,24


,missing_count,missing_percent
city,0,0.0
date,0,0.0
observed_rows,0,0.0
observed_hours,0,0.0
daily_aqi,0,0.0
aqi_valid_hours,0,0.0
pm10_mean,0,0.0
pm10_max,0,0.0
pm10_valid_hours,0,0.0
pm25_mean,0,0.0


All Notebook 02 validation checks passed.


## 11. Save and Verify the Preprocessing Outputs

The validated daily dataset is saved as `processed/daily_air_quality.csv`.

A compact JSON summary is also saved. It records the source identity, cleaning counts, study scope, aggregation rules, row counts, calendar completion results, and output columns.

The CSV is read back after saving so its dimensions, dates, cities, and city-date uniqueness can be verified.

In [11]:
daily_to_save = daily_complete.copy()
daily_to_save["date"] = daily_to_save["date"].dt.strftime("%Y-%m-%d")
daily_to_save.to_csv(OUTPUT_FILE, index=False)

preprocessing_summary = {
    "source_file": RAW_FILE.name,
    "source_sha256": source_checksum,
    "source_rows": int(source_rows),
    "source_columns": int(source_columns),
    "selected_cities": SELECTED_CITIES,
    "common_start": str(COMMON_START.date()),
    "common_end": str(COMMON_END.date()),
    "expected_dates_per_city": EXPECTED_DATES_PER_CITY,
    "selected_hourly_rows": int(len(hourly)),
    "rows_excluded_by_locked_scope": int(rows_excluded_by_scope),
    "negative_no2_to_missing": invalid_counts["negative_no2_to_missing"],
    "negative_o3_to_missing": invalid_counts["negative_o3_to_missing"],
    "co2_decision": "Dropped before daily aggregation",
    "identifier_decision": (
        "City and date retained; city_id, latitude, and longitude excluded"
    ),
    "pollutant_daily_aggregations": [
        "Mean of valid hourly readings",
        "Maximum of valid hourly readings",
    ],
    "pollutant_missing_handling": (
        "No imputation; statistics use available valid readings and "
        "valid-hour counts are retained"
    ),
    "daily_aqi_definition": "Maximum supplied hourly AQI",
    "minimum_valid_aqi_hours": MIN_VALID_AQI_HOURS,
    "observed_daily_rows_before_reindex": int(observed_daily_rows),
    "calendar_rows_after_reindex": int(len(daily_complete)),
    "inserted_calendar_days": int(inserted_calendar_days),
    "missing_daily_aqi_days": int(
        daily_complete["daily_aqi"].isna().sum()
    ),
    "created_target_or_model_features": False,
    "output_file": OUTPUT_FILE.name,
    "output_columns": list(daily_complete.columns),
    "coverage_by_city": json.loads(
        coverage_by_city.to_json(orient="records")
    ),
}

with SUMMARY_FILE.open("w", encoding="utf-8") as file:
    json.dump(
        preprocessing_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )

saved_daily = pd.read_csv(
    OUTPUT_FILE,
    parse_dates=["date"],
)

assert saved_daily.shape == daily_complete.shape
assert list(saved_daily.columns) == list(daily_complete.columns)
assert not saved_daily.duplicated(["city", "date"]).any()
assert set(saved_daily["city"]) == set(SELECTED_CITIES)
assert saved_daily["date"].min() == COMMON_START
assert saved_daily["date"].max() == COMMON_END

print("Saved:", OUTPUT_FILE)
print("Saved:", SUMMARY_FILE)
print(f"Verified saved rows: {len(saved_daily):,}")
print(f"Verified saved columns: {saved_daily.shape[1]}")
print("\nNOTEBOOK 02 PREPROCESSING GATE PASSED")

display(saved_daily.head())

Saved: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/daily_air_quality.csv
Saved: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/notebook_02_preprocessing_summary.json
Verified saved rows: 6,035
Verified saved columns: 24

NOTEBOOK 02 PREPROCESSING GATE PASSED


,city,date,observed_rows,observed_hours,daily_aqi,aqi_valid_hours,pm10_mean,pm10_max,pm10_valid_hours,pm25_mean,...,co_valid_hours,no2_mean,no2_max,no2_valid_hours,so2_mean,so2_max,so2_valid_hours,o3_mean,o3_max,o3_valid_hours
0,Bherāmāra,2022-08-05,24,24,53.0,24,18.466667,28.2,24,12.500000,...,24,16.604167,29.4,24,1.275000,3.2,24,36.958333,84.0,24
1,Bherāmāra,2022-08-06,24,24,52.0,24,18.358333,26.4,24,12.629167,...,24,17.575000,33.5,24,1.433333,4.4,24,30.916667,72.0,24
2,Bherāmāra,2022-08-07,24,24,51.0,24,16.862500,23.6,24,11.491667,...,24,16.641667,41.5,24,0.937500,1.4,24,49.958333,106.0,24
3,Bherāmāra,2022-08-08,24,24,48.0,24,11.512500,16.9,24,7.704167,...,24,8.487500,16.9,24,0.958333,1.5,24,53.916667,86.0,24
4,Bherāmāra,2022-08-09,24,24,34.0,24,8.900000,14.3,24,5.975000,...,24,6.179167,12.1,24,0.887500,2.2,24,58.166667,80.0,24


## 12. Notebook 02 Handoff

If the previous cell prints `NOTEBOOK 02 PREPROCESSING GATE PASSED`, the daily preprocessing output is ready for review.

Notebook 03 will use `daily_air_quality.csv` to:

1. Create the label for exactly the next calendar day.
2. Construct strictly historical lagged features.
3. Shift measurements before calculating rolling features.
4. Verify that no predictor contains information from the target day.

Notebook 02 deliberately contains no target, feature engineering, chronological split, or model training.